In [20]:
import kagglehub
import shutil
import os
import pandas as pd

# Download dataset dari Kaggle
# (Catatan: Pastikan path ini benar. Format standar Kaggle biasanya "username/nama-dataset")
dataset_path = kagglehub.dataset_download(
    "leonss0711/cleaned-indonesian-cyberbullying-dataset"
)

print("Dataset downloaded to:", dataset_path)

# Nama file yang mau dipakai diubah menjadi .csv
source_file = os.path.join(dataset_path, "combined_dataset.csv")
source_file2 = os.path.join(dataset_path, "kamus_singkatan.csv")

# Pindahkan ke Files Colab (/content)
destination_file = "/content/combined_dataset.csv"
destination_file2 = "/content/kamus_singkatan.csv"

shutil.copy(source_file, destination_file)
shutil.copy(source_file2, destination_file2)

print("File copied to:", destination_file)
print("File copied to:", destination_file2)

# Load CSV (parameter sep="\t" dihapus karena file sekarang dipisahkan dengan koma)
df_combined = pd.read_csv(destination_file)
df_kamus = pd.read_csv(destination_file2)

print("\nFirst 5 records of Combined Dataset:")
print(df_combined.head())

print("\nFirst 5 records of Kamus Singkatan:")
print(df_kamus.head())

Using Colab cache for faster access to the 'cleaned-indonesian-cyberbullying-dataset' dataset.
Dataset downloaded to: /kaggle/input/cleaned-indonesian-cyberbullying-dataset
File copied to: /content/combined_dataset.csv
File copied to: /content/kamus_singkatan.csv

First 5 records of Combined Dataset:
   Unnamed: 0         Label  \
0           0  Non-bullying   
1           1  Non-bullying   
2           2      Bullying   
3           3  Non-bullying   
4           4  Non-bullying   

                                          clean_text  \
0       kaka tidur yaa sudah pagi tidak boleh capek2   
1                    makan nasi padang saja badannya   
2                         suka cukur jembut manggung   
3  hai kak isyana ngefans sekali kak isyana suka ...   
4             manusia bidadari sih herann deh cantik   

                                              String  encoded_label  
0        "Kaka tidur yaa, udah pagi, gaboleh capek2"            1.0  
1            "makan nasi padang aj

In [21]:
print(f"Total data points in the dataset: {len(df_combined)}")

Total data points in the dataset: 2066


In [22]:
df = pd.read_csv(destination_file, sep="\t")

print("\nFirst 5 records:")
print(df.head())


First 5 records:
              ,Label,clean_text,String,encoded_label
0  0,Non-bullying,kaka tidur yaa sudah pagi tidak...
1  1,Non-bullying,makan nasi padang saja badannya...
2  2,Bullying,suka cukur jembut manggung,"""yang ...
3  3,Non-bullying,hai kak isyana ngefans sekali k...
4  4,Non-bullying,manusia bidadari sih herann deh...


In [23]:
print("\n--- Normalizing abbreviations in combined_dataset ---")

# Create a dictionary for abbreviation mapping from df_kamus
abbreviation_map = dict(zip(df_kamus['singkatan'], df_kamus['asli']))

def normalize_abbreviations(text, abbr_map):
    if not isinstance(text, str): # Handle non-string types, e.g., NaNs
        return text
    # Split text into words, replace abbreviations, then join back
    words = text.split()
    normalized_words = [abbr_map.get(word.lower(), word) for word in words]
    return ' '.join(normalized_words)

# Apply the normalization function to the 'clean_text' column
df_combined['clean_text_normalized'] = df_combined['clean_text'].apply(lambda x: normalize_abbreviations(x, abbreviation_map))

print("First 5 records of Combined Dataset with normalized text:")
print(df_combined[['clean_text', 'clean_text_normalized']].head())

print("Abbreviation normalization complete.")


--- Normalizing abbreviations in combined_dataset ---
First 5 records of Combined Dataset with normalized text:
                                          clean_text  \
0       kaka tidur yaa sudah pagi tidak boleh capek2   
1                    makan nasi padang saja badannya   
2                         suka cukur jembut manggung   
3  hai kak isyana ngefans sekali kak isyana suka ...   
4             manusia bidadari sih herann deh cantik   

                               clean_text_normalized  
0       kaka tidur yaa sudah pagi tidak boleh capek2  
1                    makan nasi padang saja badannya  
2                         suka cukur jembut manggung  
3  hai kak isyana ngefans sekali kak isyana suka ...  
4             manusia bidadari sih herann deh cantik  
Abbreviation normalization complete.


In [24]:
df_combined.columns

Index(['Unnamed: 0', 'Label', 'clean_text', 'String', 'encoded_label',
       'clean_text_normalized'],
      dtype='object')

In [25]:
print("\n--- Preprocessing: Removing Duplicates and Null Values ---")

print("Original shape of df_combined:", df_combined.shape)
print("Number of duplicate rows before removal:", df_combined.duplicated().sum())
print("Number of null values per column before removal:")
print(df_combined.isnull().sum())

# Remove duplicate rows
df_combined.drop_duplicates(inplace=True)
print("Shape of df_combined after removing duplicates:", df_combined.shape)

# Remove rows with any null values
df_combined.dropna(inplace=True)
print("Shape of df_combined after removing null values:", df_combined.shape)

print("Number of null values per column after removal:")
print(df_combined.isnull().sum())

print("Data cleaning (duplicates and nulls) complete.")
print("First 5 records of cleaned df_combined:")
print(df_combined.head())


--- Preprocessing: Removing Duplicates and Null Values ---
Original shape of df_combined: (2066, 6)
Number of duplicate rows before removal: 0
Number of null values per column before removal:
Unnamed: 0               0
Label                    0
clean_text               0
String                   0
encoded_label            0
clean_text_normalized    0
dtype: int64
Shape of df_combined after removing duplicates: (2066, 6)
Shape of df_combined after removing null values: (2066, 6)
Number of null values per column after removal:
Unnamed: 0               0
Label                    0
clean_text               0
String                   0
encoded_label            0
clean_text_normalized    0
dtype: int64
Data cleaning (duplicates and nulls) complete.
First 5 records of cleaned df_combined:
   Unnamed: 0         Label  \
0           0  Non-bullying   
1           1  Non-bullying   
2           2      Bullying   
3           3  Non-bullying   
4           4  Non-bullying   

                  

In [26]:
print("Unique labels in 'Label' column:")
print(df_combined['Label'].unique())

Unique labels in 'Label' column:
['Non-bullying' 'Bullying' 'positif' 'negatif' 'negative' 'positive']


In [27]:
print("\n--- Data Summary ---")
print(f"Total data points: {len(df_combined)}")

# Count the occurrences of each sentiment label
sentiment_counts = df_combined['Label'].value_counts()

print("Sentiment distribution:")
print(sentiment_counts)


--- Data Summary ---
Total data points: 2066
Sentiment distribution:
Label
negatif         614
positif         405
Bullying        325
Non-bullying    325
positive        200
negative        197
Name: count, dtype: int64


In [28]:
import pandas as pd

# Memastikan label yang dikecualikan diabaikan
excluded_labels = ['Bullying', 'Non-bullying']
df_filtered = df_combined[~df_combined['Label'].isin(excluded_labels)].copy()

# List unik dari label yang tersisa
unique_labels = df_filtered['Label'].unique()

sampled_dfs = []
for label in unique_labels:
    df_label = df_filtered[df_filtered['Label'] == label]
    # Mengambil sampel sebanyak 50 baris secara random (atau sebanyak yang ada jika kurang dari 50)
    sample_size = min(50, len(df_label))
    df_sampled = df_label.sample(n=sample_size, random_state=42)
    sampled_dfs.append(df_sampled)

# Menggabungkan semua sampel
df_final_samples = pd.concat(sampled_dfs, ignore_index=True)

# Membuat DataFrame baru dengan struktur kolom yang diinginkan
# 'sentiment' diisi dari 'clean_text_normalized' (atau 'clean_text')
# 'label' diisi dari 'Label'
# 'B/N' dibiarkan kosong
export_df = pd.DataFrame({
    'sentiment': df_final_samples['clean_text_normalized'],
    'label': df_final_samples['Label'],
    'B/N': ''
})

# Simpan ke file Excel
output_filename = '/content/sentiment_samples.xlsx'
export_df.to_excel(output_filename, index=False)

print(f"File Excel berhasil dibuat di: {output_filename}")
print(f"Total data yang diekspor: {len(export_df)} baris.")
print("Distribusi label pada file:")
print(export_df['label'].value_counts())


File Excel berhasil dibuat di: /content/sentiment_samples.xlsx
Total data yang diekspor: 200 baris.
Distribusi label pada file:
label
positif     50
negatif     50
negative    50
positive    50
Name: count, dtype: int64


In [29]:
import pandas as pd
from google.colab import files
from sklearn.metrics import cohen_kappa_score

# 1. Proses upload file pertama dari komputer
print("Silakan unggah file Excel pertama (misalnya anotasi dari Rater 1 / Marco):")
uploaded_1 = files.upload()
file1_name = list(uploaded_1.keys())[0]
print(f"Berhasil mengunggah file pertama: {file1_name}\n")

# 2. Proses upload file kedua dari komputer
print("Silakan unggah file Excel kedua (misalnya anotasi dari Rater 2 / Vuitton):")
uploaded_2 = files.upload()
file2_name = list(uploaded_2.keys())[0]
print(f"Berhasil mengunggah file kedua: {file2_name}\n")

try:
    # Membaca kedua file yang diunggah
    df1 = pd.read_excel(file1_name)
    df2 = pd.read_excel(file2_name)

    # Memastikan kolom 'B/N' ada di kedua file
    if 'B/N' not in df1.columns or 'B/N' not in df2.columns:
        print("Error: Kolom 'B/N' tidak ditemukan pada salah satu atau kedua file.")
    else:
        # Membersihkan spasi, menangani NaN, dan menyamakan ke huruf kapital
        df1['B/N'] = df1['B/N'].fillna('KOSONG').astype(str).str.strip().str.upper()
        df2['B/N'] = df2['B/N'].fillna('KOSONG').astype(str).str.strip().str.upper()

        # Menggabungkan data berdasarkan indeks untuk perbandingan sejajar
        comparison_df = pd.DataFrame({
            'Rater_1': df1['B/N'],
            'Rater_2': df2['B/N']
        })

        # Menghitung Percentage Agreement
        total_rows = len(comparison_df)
        matching_rows = (comparison_df['Rater_1'] == comparison_df['Rater_2']).sum()
        percentage_agreement = (matching_rows / total_rows) * 100 if total_rows > 0 else 0

        # Menghitung Cohen's Kappa
        kappa_score = cohen_kappa_score(comparison_df['Rater_1'], comparison_df['Rater_2'])

        # Tampilkan Hasil
        print("=== HASIL PERBANDINGAN ANOTASI DARI COMPUTER ===")
        print(f"File 1                          : {file1_name}")
        print(f"File 2                          : {file2_name}")
        print(f"Total data yang dibandingkan    : {total_rows} baris")
        print(f"Jumlah kecocokan data (match)   : {matching_rows} baris")
        print(f"Percentage Agreement            : {percentage_agreement:.2f}%")
        print(f"Cohen's Kappa Score             : {kappa_score:.4f}")

        # Interpretasi Cohen's Kappa
        print("\nInterpretasi Nilai Cohen's Kappa:")
        if kappa_score < 0:
            print("- Kurang dari 0: Tidak ada kesepakatan (No agreement)")
        elif kappa_score <= 0.20:
            print("- 0.00 - 0.20  : Kesepakatan sangat lemah (Slight agreement)")
        elif kappa_score <= 0.40:
            print("- 0.21 - 0.40  : Kesepakatan cukup (Fair agreement)")
        elif kappa_score <= 0.60:
            print("- 0.41 - 0.60  : Kesepakatan sedang (Moderate agreement)")
        elif kappa_score <= 0.80:
            print("- 0.61 - 0.80  : Kesepakatan substansial (Substantial agreement)")
        else:
            print("- 0.81 - 1.00  : Kesepakatan hampir sempurna (Almost perfect agreement)")

except Exception as e:
    print(f"Terjadi kesalahan saat memproses file: {e}")


Silakan unggah file Excel pertama (misalnya anotasi dari Rater 1 / Marco):


Saving SA Reyhan.xlsx to SA Reyhan (2).xlsx
Berhasil mengunggah file pertama: SA Reyhan (2).xlsx

Silakan unggah file Excel kedua (misalnya anotasi dari Rater 2 / Vuitton):


Saving SA Marco.xlsx to SA Marco (6).xlsx
Berhasil mengunggah file kedua: SA Marco (6).xlsx

=== HASIL PERBANDINGAN ANOTASI DARI COMPUTER ===
File 1                          : SA Reyhan (2).xlsx
File 2                          : SA Marco (6).xlsx
Total data yang dibandingkan    : 200 baris
Jumlah kecocokan data (match)   : 152 baris
Percentage Agreement            : 76.00%
Cohen's Kappa Score             : 0.5349

Interpretasi Nilai Cohen's Kappa:
- 0.41 - 0.60  : Kesepakatan sedang (Moderate agreement)


In [30]:
import pandas as pd
import numpy as np

try:
    # 1. Mengambil data dari variabel yang sudah didefinisikan di memori
    r1 = df1['B/N'].fillna('KOSONG').astype(str).str.strip().str.upper()
    r2 = df2['B/N'].fillna('KOSONG').astype(str).str.strip().str.upper()
    r3 = df3['B/N'].fillna('KOSONG').astype(str).str.strip().str.upper()

    df_all = pd.DataFrame({
        'Reyhan': r1,
        'Marco': r2,
        'Vuitton': r3
    })

    # Memastikan semua rater menggunakan kategori yang sama (B dan N)
    # Mengabaikan atau menandai label di luar B dan N sebagai KOSONG
    valid_categories = {'B', 'N'}
    for col in df_all.columns:
        df_all[col] = df_all[col].apply(lambda x: x if x in valid_categories else 'KOSONG')

    total_rows = len(df_all)

    # 2. Hitung Overall Exact Agreement (3 rater setuju 100% pada pilihan yang sama)
    exact_matches = ((df_all['Reyhan'] == df_all['Marco']) & (df_all['Marco'] == df_all['Vuitton'])).sum()
    overall_agreement_pct = (exact_matches / total_rows) * 100

    # 3. Hitung Fleiss' Kappa secara manual untuk menghindari error statsmodels pada kasus asimetris
    # Fleiss' Kappa formula: kappa = (p_bar - p_e_bar) / (1 - p_e_bar)
    N_subjects = total_rows
    n_raters = 3
    categories = ['B', 'N', 'KOSONG']
    k_categories = len(categories)

    # Buat tabel kontingensi (subjek x kategori)
    cont_matrix = np.zeros((N_subjects, k_categories))
    cat_to_idx = {cat: idx for idx, cat in enumerate(categories)}

    for i in range(N_subjects):
        for rater in ['Reyhan', 'Marco', 'Vuitton']:
            val = df_all.at[i, rater]
            idx = cat_to_idx[val]
            cont_matrix[i, idx] += 1

    # p_i: proporsi kesepakatan untuk setiap subjek
    p_i = (np.sum(cont_matrix ** 2, axis=1) - n_raters) / (n_raters * (n_raters - 1))
    p_bar = np.mean(p_i)

    # p_j: proporsi semua penilaian yang masuk ke kategori j
    p_j = np.sum(cont_matrix, axis=0) / (N_subjects * n_raters)
    p_e_bar = np.sum(p_j ** 2)

    if p_e_bar < 1.0:
        f_kappa = (p_bar - p_e_bar) / (1 - p_e_bar)
    else:
        f_kappa = 1.0

    # Tampilkan Hasil Analisis Gabungan yang Diperbaiki
    print("=======================================================")
    print("=== ANALISIS KESEPAKATAN MULTI-RATER (3 ANNOTATOR) ===")
    print("=======================================================")
    print(f"Total Baris Data                : {total_rows}")
    print(f"Kecocokan Sempurna (3 Setuju)   : {exact_matches} baris")
    print(f"Overall Exact Agreement         : {overall_agreement_pct:.2f}%")
    print(f"Fleiss' Kappa Score             : {f_kappa:.4f}")

    print("\nInterpretasi Nilai Fleiss' Kappa:")
    if f_kappa < 0:
        print("- Kurang dari 0: Tidak ada kesepakatan (Poor agreement)")
    elif f_kappa <= 0.20:
        print("- 0.00 - 0.20  : Kesepakatan sangat lemah (Slight agreement)")
    elif f_kappa <= 0.40:
        print("- 0.21 - 0.40  : Kesepakatan cukup (Fair agreement)")
    elif f_kappa <= 0.60:
        print("- 0.41 - 0.60  : Kesepakatan sedang (Moderate agreement)")
    elif f_kappa <= 0.80:
        print("- 0.61 - 0.80  : Kesepakatan substansial (Substantial agreement)")
    else:
        print("- 0.81 - 1.00  : Kesepakatan hampir sempurna (Almost perfect agreement)")

except Exception as e:
    print(f"Terjadi kesalahan saat memproses data: {e}")

=== ANALISIS KESEPAKATAN MULTI-RATER (3 ANNOTATOR) ===
Total Baris Data                : 200
Kecocokan Sempurna (3 Setuju)   : 142 baris
Overall Exact Agreement         : 71.00%
Fleiss' Kappa Score             : 0.6118

Interpretasi Nilai Fleiss' Kappa:
- 0.61 - 0.80  : Kesepakatan substansial (Substantial agreement)


In [31]:
import pandas as pd
import numpy as np
from google.colab import files

print("=== PROSES UPLOAD 3 FILE ANOTASI ===\n")

# 1. Upload File Rater 1 (Reyhan)
print("1. Unggah file Excel untuk Rater 1 (Reyhan):")
uploaded_r1 = files.upload()
file_r1_name = list(uploaded_r1.keys())[0]
print(f"Berhasil mengunggah file Reyhan: {file_r1_name}\n")

# 2. Upload File Rater 2 (Marco)
print("2. Unggah file Excel untuk Rater 2 (Marco):")
uploaded_r2 = files.upload()
file_r2_name = list(uploaded_r2.keys())[0]
print(f"Berhasil mengunggah file Marco: {file_r2_name}\n")

# 3. Upload File Rater 3 (Vuitton)
print("3. Unggah file Excel untuk Rater 3 (Vuitton):")
uploaded_r3 = files.upload()
file_r3_name = list(uploaded_r3.keys())[0]
print(f"Berhasil mengunggah file Vuitton: {file_r3_name}\n")

try:
    # Membaca ketiga file yang diunggah
    df1 = pd.read_excel(file_r1_name)
    df2 = pd.read_excel(file_r2_name)
    df3 = pd.read_excel(file_r3_name)

    # Membersihkan dan menyamakan format label 'B/N'
    r1 = df1['B/N'].fillna('KOSONG').astype(str).str.strip().str.upper()
    r2 = df2['B/N'].fillna('KOSONG').astype(str).str.strip().str.upper()
    r3 = df3['B/N'].fillna('KOSONG').astype(str).str.strip().str.upper()

    df_all = pd.DataFrame({
        'Reyhan': r1,
        'Marco': r2,
        'Vuitton': r3
    })

    # Memastikan kategori yang valid adalah B, N, dan KOSONG
    valid_categories = {'B', 'N'}
    for col in df_all.columns:
        df_all[col] = df_all[col].apply(lambda x: x if x in valid_categories else 'KOSONG')

    total_rows = len(df_all)

    # Hitung Overall Exact Agreement (3 rater setuju 100% pada pilihan yang sama)
    exact_matches = ((df_all['Reyhan'] == df_all['Marco']) & (df_all['Marco'] == df_all['Vuitton'])).sum()
    overall_agreement_pct = (exact_matches / total_rows) * 100

    # Hitung Fleiss' Kappa
    N_subjects = total_rows
    n_raters = 3
    categories = ['B', 'N', 'KOSONG']
    k_categories = len(categories)

    cont_matrix = np.zeros((N_subjects, k_categories))
    cat_to_idx = {cat: idx for idx, cat in enumerate(categories)}

    for i in range(N_subjects):
        for rater in ['Reyhan', 'Marco', 'Vuitton']:
            val = df_all.at[i, rater]
            idx = cat_to_idx[val]
            cont_matrix[i, idx] += 1

    p_i = (np.sum(cont_matrix ** 2, axis=1) - n_raters) / (n_raters * (n_raters - 1))
    p_bar = np.mean(p_i)

    p_j = np.sum(cont_matrix, axis=0) / (N_subjects * n_raters)
    p_e_bar = np.sum(p_j ** 2)

    if p_e_bar < 1.0:
        f_kappa = (p_bar - p_e_bar) / (1 - p_e_bar)
    else:
        f_kappa = 1.0

    # Tampilkan Hasil Analisis
    print("=======================================================")
    print("=== ANALISIS KESEPAKATAN MULTI-RATER (3 ANNOTATOR) ===")
    print("=======================================================")
    print(f"File Reyhan                    : {file_r1_name}")
    print(f"File Marco                     : {file_r2_name}")
    print(f"File Vuitton                   : {file_r3_name}")
    print(f"Total Baris Data                : {total_rows}")
    print(f"Kecocokan Sempurna (3 Setuju)   : {exact_matches} baris")
    print(f"Overall Exact Agreement         : {overall_agreement_pct:.2f}%")
    print(f"Fleiss' Kappa Score             : {f_kappa:.4f}")

    print("\nInterpretasi Nilai Fleiss' Kappa:")
    if f_kappa < 0:
        print("- Kurang dari 0: Tidak ada kesepakatan (Poor agreement)")
    elif f_kappa <= 0.20:
        print("- 0.00 - 0.20  : Kesepakatan sangat lemah (Slight agreement)")
    elif f_kappa <= 0.40:
        print("- 0.21 - 0.40  : Kesepakatan cukup (Fair agreement)")
    elif f_kappa <= 0.60:
        print("- 0.41 - 0.60  : Kesepakatan sedang (Moderate agreement)")
    elif f_kappa <= 0.80:
        print("- 0.61 - 0.80  : Kesepakatan substansial (Substantial agreement)")
    else:
        print("- 0.81 - 1.00  : Kesepakatan hampir sempurna (Almost perfect agreement)")

except Exception as e:
    print(f"Terjadi kesalahan saat memproses data: {e}")

=== PROSES UPLOAD 3 FILE ANOTASI ===

1. Unggah file Excel untuk Rater 1 (Reyhan):


Saving SA Reyhan.xlsx to SA Reyhan (3).xlsx
Berhasil mengunggah file Reyhan: SA Reyhan (3).xlsx

2. Unggah file Excel untuk Rater 2 (Marco):


Saving SA Marco.xlsx to SA Marco (7).xlsx
Berhasil mengunggah file Marco: SA Marco (7).xlsx

3. Unggah file Excel untuk Rater 3 (Vuitton):


Saving SA Vuitton.xlsx to SA Vuitton (5).xlsx
Berhasil mengunggah file Vuitton: SA Vuitton (5).xlsx

=== ANALISIS KESEPAKATAN MULTI-RATER (3 ANNOTATOR) ===
File Reyhan                    : SA Reyhan (3).xlsx
File Marco                     : SA Marco (7).xlsx
File Vuitton                   : SA Vuitton (5).xlsx
Total Baris Data                : 200
Kecocokan Sempurna (3 Setuju)   : 142 baris
Overall Exact Agreement         : 71.00%
Fleiss' Kappa Score             : 0.6118

Interpretasi Nilai Fleiss' Kappa:
- 0.61 - 0.80  : Kesepakatan substansial (Substantial agreement)


In [34]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

try:
    # Gunakan data dari df_all (yang menampung Reyhan, Marco, Vuitton) dan df1/df2 untuk mengambil label aslinya
    # Kita buat DataFrame baru untuk analisis perbandingan ini
    analysis_df = pd.DataFrame({
        'sentiment': df1['sentiment'],
        'original_label': df1['label'],
        'Rater_Reyhan': df_all['Reyhan'],
        'Rater_Marco': df_all['Marco'],
        'Rater_Vuitton': df_all['Vuitton']
    })

    # 1. Membuat mapped_label
    # positif & positive -> Non-Bullying
    # negatif & negative -> Bullying
    def map_original_label(label):
        label_clean = str(label).strip().lower()
        if label_clean in ['positif', 'positive']:
            return 'Non-Bullying'
        elif label_clean in ['negatif', 'negative']:
            return 'Bullying'
        else:
            return 'Unknown'

    analysis_df['mapped_label'] = analysis_df['original_label'].apply(map_original_label)

    # 2. Menentukan real_label berdasarkan mayoritas suara dari 3 anotator
    # B -> Bullying, N -> Non-Bullying
    def determine_real_label(row):
        votes = [row['Rater_Reyhan'], row['Rater_Marco'], row['Rater_Vuitton']]
        # Hitung jumlah B dan N
        b_count = votes.count('B')
        n_count = votes.count('N')

        if b_count > n_count:
            return 'Bullying'
        else:
            return 'Non-Bullying'

    analysis_df['real_label'] = analysis_df.apply(determine_real_label, axis=1)

    # 3. Hitung Percentage Agreement dan Cohen's Kappa antara mapped_label dan real_label
    total_data = len(analysis_df)
    matches = (analysis_df['mapped_label'] == analysis_df['real_label']).sum()
    percentage_agreement = (matches / total_data) * 100

    kappa_val = cohen_kappa_score(analysis_df['mapped_label'], analysis_df['real_label'])

    # Tampilkan beberapa sampel data hasil mapping dan penentuan real_label
    print("=== CONTOH DATA HASIL MAPPING DAN REAL_LABEL (10 BARIS PERTAMA) ===")
    print(analysis_df[['sentiment', 'original_label', 'mapped_label', 'Rater_Reyhan', 'Rater_Marco', 'Rater_Vuitton', 'real_label']].head(10))
    print("\n" + "="*60)
    print("=== METRIK EVALUASI: MAPPED LABEL vs REAL LABEL ===")
    print("="*60)
    print(f"Total Data                : {total_data} baris")
    print(f"Jumlah Kesocokan (Match)   : {matches} baris")
    print(f"Percentage Agreement      : {percentage_agreement:.2f}%")
    print(f"Cohen's Kappa Score       : {kappa_val:.4f}")

    print("\nInterpretasi Nilai Cohen's Kappa:")
    if kappa_val < 0:
        print("- Kurang dari 0: Tidak ada kesepakatan (No agreement)")
    elif kappa_val <= 0.20:
        print("- 0.00 - 0.20  : Kesepakatan sangat lemah (Slight agreement)")
    elif kappa_val <= 0.40:
        print("- 0.21 - 0.40  : Kesepakatan cukup (Fair agreement)")
    elif kappa_val <= 0.60:
        print("- 0.41 - 0.60  : Kesepakatan sedang (Moderate agreement)")
    elif kappa_val <= 0.80:
        print("- 0.61 - 0.80  : Kesepakatan substansial (Substantial agreement)")
    else:
        print("- 0.81 - 1.00  : Kesepakatan hampir sempurna (Almost perfect agreement)")

except Exception as e:
    print(f"Terjadi kesalahan: {e}. Pastikan data rater sebelumnya sudah ter-load dengan benar.")

=== CONTOH DATA HASIL MAPPING DAN REAL_LABEL (10 BARIS PERTAMA) ===
                                           sentiment original_label  \
0          kecuali manusia monyet mamalia buta warna        positif   
1  cewenya manis cakepan yang sekarang dari yang ...        positif   
2                          babi pintar mencari makan        positif   
3  guinness book of world records anjing terkecil...        positif   
4  anak anak anjing dalmatian terlahir murni puti...        positif   
5  saya ikuti heart series monyet cantik saja yan...        positif   
6              gileee suaraaanyaa kereen parah sukak        positif   
7               senang hasil rapat keputusan diambil        positif   
8                 kamu berhasil suasana terima kasih        positif   
9  benar pepatah yess buah jatuh tidak pohon nya ...        positif   

   mapped_label Rater_Reyhan Rater_Marco Rater_Vuitton    real_label  
0  Non-Bullying            N           N             B  Non-Bullying  
1  Non-B